# 解答③ HuggingFace で LLM 推論パラメータを探索する

> **このファイルは `exercises/ex_03_llm_inference.ipynb` の解答です。**  
> 課題に取り組む前に見ないようにしてください。

---

## セットアップ

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from itertools import combinations
from transformers import AutoTokenizer, AutoModelForCausalLM

if os.path.exists('/data/shared/models'):
    os.environ['HF_HOME'] = '/data/shared/models'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')
torch.manual_seed(42)

---

## Step 1: モデルの読み込み

In [ ]:
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('モデルロード完了')

---

## Step 2: generate_text() 関数の実装（解答）

In [ ]:
def build_prompt(user_message: str, system: str = 'You are a helpful assistant.') -> str:
    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': user_message},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_text(
    prompt: str,
    max_new_tokens: int = 150,
    do_sample: bool = True,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
) -> str:
    # 解答: トークナイズ → device へ移動
    inputs    = tokenizer(prompt, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    # 解答: no_grad ブロック内で generate() を呼び出す
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 解答: プロンプト部分を除いてデコード
    generated_ids = outputs[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


question = '深層学習を一言で説明してください。'
response = generate_text(build_prompt(question))
print(f'generate_text() ✓')
print(f'質問: {question}')
print(f'回答: {response}')

---

## Step 3: Temperature の実験（解答）

In [ ]:
def type_token_ratio(text: str) -> float:
    # 解答: encode してユニーク比率を返す
    token_ids = tokenizer.encode(text)
    return len(set(token_ids)) / len(token_ids) if token_ids else 0.0


prompt_temp  = build_prompt('AIの未来について、自由に語ってください。')
temperatures = [0.1, 0.7, 1.3]
colors       = ['#0D4A38', '#7C5C00', '#991B1B']
temp_results = {}

# 解答: 各 temperature で生成
for t in temperatures:
    torch.manual_seed(42)
    response = generate_text(prompt_temp, max_new_tokens=120, temperature=t, top_p=0.9)
    temp_results[t] = response
    print(f'\n{"="*60}')
    print(f'temperature = {t}')
    print('='*60)
    print(response)

# 解答: TTR を計算して棒グラフ
ttrs = {t: type_token_ratio(resp) for t, resp in temp_results.items()}

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar([str(t) for t in temperatures], list(ttrs.values()), color=colors, width=0.5)
for bar, val in zip(bars, ttrs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_xlabel('temperature')
ax.set_ylabel('Type-Token Ratio（高いほど多様）')
ax.set_title('Temperature とトークン多様性の関係')
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

---

## Step 4: Top-p の実験（解答）

In [ ]:
prompt_topp = build_prompt('料理のレシピを1つ考えてください。')
top_ps      = [0.5, 0.9, 1.0]

# 解答
for p in top_ps:
    torch.manual_seed(42)
    response = generate_text(prompt_topp, max_new_tokens=120, temperature=0.9, top_p=p)
    print(f'\n{"="*60}')
    print(f'top_p = {p}')
    print('='*60)
    print(response)

---

## Step 5: 複数回生成のばらつきを可視化（解答）

In [ ]:
def token_overlap(text_a: str, text_b: str) -> float:
    # 解答: Jaccard 類似度
    set_a = set(tokenizer.encode(text_a))
    set_b = set(tokenizer.encode(text_b))
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)


N_TRIALS   = 5
prompt_var = build_prompt('今日の気分を一文で表現してください。')
test_temps = [0.1, 0.7, 1.3]
colors_var = ['#0D4A38', '#7C5C00', '#991B1B']

# 解答: 各 temperature で N_TRIALS 回生成
all_outputs = {}
for t in test_temps:
    outputs = []
    for i in range(N_TRIALS):
        torch.manual_seed(i)
        resp = generate_text(prompt_var, max_new_tokens=60, temperature=t, top_p=0.9)
        outputs.append(resp)
    all_outputs[t] = outputs
    print(f'temperature={t}: {N_TRIALS}回生成完了')

# 解答: Jaccard 平均を計算して棒グラフ
avg_overlaps = {}
for t, outputs in all_outputs.items():
    pairs = list(combinations(outputs, 2))
    avg_overlaps[t] = sum(token_overlap(a, b) for a, b in pairs) / len(pairs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([str(t) for t in test_temps], list(avg_overlaps.values()), color=colors_var, width=0.5)
for i, val in enumerate(avg_overlaps.values()):
    axes[0].text(i, val + 0.005, f'{val:.3f}', ha='center', fontsize=10)
axes[0].set_xlabel('temperature')
axes[0].set_ylabel('平均 Jaccard 類似度（低いほど多様）')
axes[0].set_title('5回生成の平均ばらつき')
axes[0].set_ylim(0, 1.0)

axes[1].axis('off')
t_show = 0.7
text_lines = [f'temperature = {t_show}　各回の出力:\n']
for i, resp in enumerate(all_outputs[t_show]):
    short = resp[:60] + ('…' if len(resp) > 60 else '')
    text_lines.append(f'[{i+1}] {short}')
axes[1].text(0.02, 0.95, '\n'.join(text_lines),
             transform=axes[1].transAxes, fontsize=8, va='top', ha='left',
             bbox=dict(boxstyle='round', facecolor='#F0EDE6', alpha=0.8))

plt.tight_layout()
plt.show()

---

## チャレンジ問題の解答: repetition_penalty

In [ ]:
prompt_rep   = build_prompt('春について、詩のように語ってください。')
rep_penalties = [1.0, 1.2, 1.5]

for r in rep_penalties:
    torch.manual_seed(42)
    response  = generate_text(
        prompt_rep, max_new_tokens=150, temperature=0.9, top_p=0.92, repetition_penalty=r
    )
    # 解答: 2-gram 重複率の計算
    token_ids = tokenizer.encode(response)
    bigrams   = list(zip(token_ids, token_ids[1:]))
    dup_ratio = 1 - len(set(bigrams)) / max(len(bigrams), 1)

    print(f'\n{"="*60}')
    print(f'repetition_penalty = {r}  |  2-gram 重複率: {dup_ratio:.3f}')
    print('='*60)
    print(response)